# Manual Pipeline 2
- In this notebook, I will refactor the manual_pipeline1 processing into a single function.
- I will have an option for autopower or n_points as part of the function.
- Add function that tests for closeness between selected peaks:
    - Consider the LSP resolution.
- Afterwards cluster peaks in the same ranges and choose the strongest like in the priors analysis.

In [2]:
import pandas as pd
from astropy.time import Time
import matplotlib.pyplot as plt
import celerite2
import numpy as np
from celerite2 import terms
from scipy.optimize import minimize
from prettytable import PrettyTable
from astropy.timeseries import LombScargle
from scipy.signal import find_peaks
from scipy.optimize import curve_fit


# Importing data
data = pd.read_csv(r"./Data/benchmark/HD81809_Mt_wilson_data.txt", sep='\s',skip_blank_lines=True)

# data = pd.read_csv(r"./Data/hd166_caii.txt", sep='\s',skip_blank_lines=True)
data.columns = ["JD", "sind"]
data['JD'] = data['JD'] + 2400000.0 # this file drops the 24 prefix
time_obj = Time(data["JD"].to_numpy(), format='jd', scale='tdb')
data["year"] = time_obj.jyear
data["day"] = time_obj.jd 
data['datetime'] = time_obj.to_datetime(timezone=None)
data = data.set_index('datetime')

# No need to down sample the dataset here. yay!
def split_df(df, train_split=0.8, valid_split=0.19):
    n = len(df)
    train_idx = round(train_split * n)
    valid_idx = round((train_split + valid_split) * n)
    return df.iloc[:train_idx].copy(), df.iloc[train_idx:valid_idx].copy(), df.iloc[valid_idx:].copy()

# Split the data
train_df, valid_df, test_df = split_df(data)


C:\Users\Joey\AppData\Local\Temp\ipykernel_27420\3617134450.py:15: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  data = pd.read_csv(r"./Data/benchmark/HD81809_Mt_wilson_data.txt", sep='\s',skip_blank_lines=True)


In [ ]:
def find_peaks(df, manual_freq = None, period_range = [0.1, 100*365], n_periods = 10000, FAPs = [10,5,1,0.1], key_FAP_idx = -1, plot_LSP = False, plot_resids = False):
    '''
    Iteratively finds the peaks of the LSP.
    This will be used in the prior selsection process.
    Finds one peak then checks if it is an alias or a window.
    Also checks if it is too similar to a previous peak for the LSP to have resolved.
    If that peak also satiesfies an SNR threshold, add it to an accepted periods list.

    df is the dataframe being processed (training set)
    manual_freq True means the period_range and n_periods defines the set of periods calculated.
    Period_range is in days

    Returns a list of accepted periods.
    '''
    accepted_peak_freqs = [] # We will work in frequency space for the analysis other than for the plotting. It is easier for the SNR fitting.

    if manual_freq == 'linear':
        min_period = period_range[0]
        max_period = period_range[1]
        periods = np.linspace(min_period, max_period, n_periods)
        freqs = 1 / periods
    if manual_freq == 'log':
        min_period = period_range[0]
        max_period = period_range[1]
        periods = np.logspace(np.log(min_period), np.log(max_period), n_periods)
        freqs = 1 / periods
    
    resids = [df['sind']]
    i = 0 # What index in resids we are presently processing
    t = df['day'] # Fixed time axis for al

    while True:
        # Take the LSP : powers for the freqs
        ls = LombScargle(t, resids[i])
        if manual_freq is not None: # Use autopower
            freqs, powers = ls.autopower()
        else:
            powers = ls.power(freqs)

        # Now calculate the FAPs
        FAPs = np.array(FAPs)/100
        power_invFAPs = ls.false_alarm_level(FAPs, method = 'bootstrap') 
        key_FAP = FAPs[key_FAP_idx]

        # Iden
    
        
        
    
    
    

